# Introduction et Contexte Métier {#sec-intro}

**Problématique** : Comment prédire la valeur marchande future d'un joueur et automatiser la détection de son profil de jeu sur le terrain ?
**Le Contexte Métier** : Les clubs cherchent à anticiper la plus-value financière d'un joueur avant qu'il n'explose sur la scène internationale (achat à bas coût, revente élevée).

## Contexte du Projet

 Le football professionnel est devenu une industrie standardisée où les données guident les investissements financiers. L'identification de talents (Scouting) et l'estimation de la valeur marchande des joueurs représentent des enjeux de plusieurs dizaines de millions d'euros pour les clubs, à l'image des stratégies menées par des structures modernes comme le LOSC au sein de la métropole lilloise. Ce projet propose une approche quantitative et multi-source pour optimiser le processus de recrutement. En combinant l'historique des performances sportives issues de la plateforme Transfermarkt et l'analyse automatisée des flux vidéo d'action par vision par ordinateur, nous cherchons à transformer des signaux faibles athlétiques et techniques en indicateurs financiers fiables pour les décideurs sportifs.

## Objectif Analytique

 L'objectif principal de ce projet est de concevoir un pipeline de données hybride répondant à deux tâches distinctes et complémentaires :

 Modélisation Tabulaire (Machine Learning) : Développer un modèle de régression (ex: XGBoost, Random Forest) visant à prédire la valeur marchande (market_value_in_eur) d'un joueur en fonction de ses caractéristiques intrinsèques, de ses statistiques de performance cumulées et du contexte compétitif de son club.

 Modélisation Vision (Deep Learning CNN) : Concevoir un réseau de neurones convolutif (CNN) sous TensorFlow/Keras capable de classifier le type d'action technique (Tackle, Shoot, Pass) effectué par un joueur à partir de captures d'écran de matchs.

 Le couplage de ces deux approches permet d'enrichir les profils des joueurs par des métriques comportementales extraites directement de la vidéo, offrant ainsi un livrable analytique complet sous forme de tableau de bord d'aide à la décision pour le recrutement prédictif.

---

# Acquisition et Préparation des Données (Data Wrangling) {#sec-wrangling}

Le succès de tout projet de Data Science repose sur la qualité de la préparation des données [@pandas2020]. Cette section documente l'audit de qualité et les étapes de nettoyage appliquées à vos jeux de données bruts.

## Chapitre 1 : Acquisition Multi-Sources


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 📥 Étape 1 : Acquisition des Données & Multi-Sources (Squelette Étudiant)

Cette étape correspond au premier chapitre du pipeline de Data Science. L'objectif est d'identifier, d'importer et de consolider vos jeux de données bruts issus de différentes sources (fichiers CSV locaux, requêtes API, bases de données, etc.).

### 1. Initialisation de l'environnement


In [ ]:
import os
import shutil
import kagglehub
import pandas as pd

### 2. Téléchargement via Kagglehub


In [ ]:
print("Démarrage du téléchargement du dataset Transfermarkt...")
# Télécharge la dernière version stable dans le cache local de kagglehub
cache_path = kagglehub.dataset_download("davidcariboo/player-scores")
print(f"Fichiers temporaires téléchargés par kagglehub dans : {cache_path}")

### 3. Structuration et persistance locale


In [ ]:
target_dir = "../data/raw/"
os.makedirs(target_dir, exist_ok=True)

print(f"Copie des fichiers essentiels vers le dossier local du projet ({target_dir})...")
files_to_copy = ["players.csv", "appearances.csv", "clubs.csv", "competitions.csv"]

for file_name in os.listdir(cache_path):
    if file_name in files_to_copy:
        src_file = os.path.join(cache_path, file_name)
        dst_file = os.path.join(target_dir, file_name)
        shutil.copy(src_file, dst_file)
        print(f" -> {file_name} copié avec succès.")

### 4. Vérification 


In [ ]:
print("\n--- Vérification du chargement des données ---")

# Lecture d'un échantillon pour valider que tout fonctionne
df_players = pd.read_csv(os.path.join(target_dir, "players.csv"))
df_appearances = pd.read_csv(os.path.join(target_dir, "appearances.csv"))

print(f"Table 'players' chargée avec succès : {df_players.shape[0]} lignes, {df_players.shape[1]} colonnes.")
print(f"Table 'appearances' chargée avec succès : {df_appearances.shape[0]} lignes, {df_appearances.shape[1]} colonnes.")

# Forcer Pandas à afficher TOUTES les colonnes
pd.set_option('display.max_columns', None)

# Affichage des premières lignes pour validation visuelle
df_players.head(10)

## Chapitre 2 : Nettoyage et Préparation (Wrangling)


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 🧹 Étape 2 : Préparation & Nettoyage de Données (Data Wrangling) (Squelette Étudiant)

Cette étape correspond au deuxième chapitre du projet. L'objectif est d'effectuer un audit de qualité de vos données brutes, puis de mettre en œuvre un nettoyage rigoureux à l'aide de votre package personnalisé `src.data_clean`.

### 1. Importations et configuration du chemin src


In [ ]:
import os
import sys
import pandas as pd

# Ajout du dossier racine au chemin pour pouvoir importer le module src
sys.path.append(os.path.abspath(os.path.join('..')))
import src.data_clean as dc 

### 2. Chargement du dataset brut et Audit Initial

**À COMPLÉTER PAR L'ÉTUDIANT :**
Chargez les données brutes et inspectez la qualité du dataset (taux de valeurs manquantes, présence de doublons, types erronés).


In [ ]:
print("Chargement des données brutes...")
df_raw_players = pd.read_csv("../data/raw/players.csv")
df_raw_appearances = pd.read_csv("../data/raw/appearances.csv")

### 3. Nettoyage et Imputation


In [ ]:
print("Application des filtres et imputations...")
# 1. Traitement des dates
df_cleaned_players = dc.clean_dates(df_raw_players, ['date_of_birth', 'last_season'])

# 2. Calcul de l'âge (Variable cruciale dérivée de la date de naissance)
current_year = pd.Timestamp.now().year
df_cleaned_players['age'] = current_year - df_cleaned_players['date_of_birth'].dt.year
# Remplacement des âges aberrants si présents par la médiane
df_cleaned_players['age'] = df_cleaned_players['age'].fillna(df_cleaned_players['age'].median())

# 3. Imputation des valeurs manquantes (Taille, Pied fort, Valeur marchande)
df_cleaned_players = dc.impute_missing_data(df_cleaned_players)

### 4. Agrégation des performances de matchs


In [ ]:
print("Agrégation des statistiques de matchs par joueur...")
df_perf_aggregated = dc.aggregate_performances(df_raw_appearances)

### 5. Fusion finale et Sauvegarde


In [ ]:
print("Fusion des sources et création du dataset d'apprentissage...")
df_final = dc.build_final_dataset(df_cleaned_players, df_perf_aggregated)

# Création du dossier processed s'il n'existe pas
os.makedirs("../data/processed/", exist_ok=True)

# Sauvegarde au format CSV pour les étapes d'EDA et de Modélisation
output_path = "../data/processed/football_ml_dataset.csv"
df_final.to_csv(output_path, index=False)

print(f"\n[SUCCÈS] Data Wrangling terminé !")
print(f"Le dataset propre contient {df_final.shape[0]} joueurs et {df_final.shape[1]} caractéristiques.")
print(f"Fichier sauvegardé dans : {output_path}")

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)

# Chargement du fichier traité
df_processed = pd.read_csv("../data/processed/football_ml_dataset.csv")

# Affichage des 5 premières lignes sous forme de joli tableau HTML interactif
df_processed.head(10)

# Visualisation Multidimensionnelle (Insights) {#sec-viz}

Nous présentons ici les résultats visuels clés permettant de dégager des insights exploitables pour les décideurs, en s'appuyant sur notre module `src/utils_viz.py`.

## Chapitre 3 : Travaux Pratiques d'Exploration Visuelle


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 📊 Étape 4 : Visualisation Multidimensionnelle (Squelette Étudiant)

Cette étape correspond au quatrième chapitre du cours. L'objectif est de concevoir des représentations visuelles premium pour identifier des tendances et insights clés à l'aide de votre package personnalisé de tracé `src.utils_viz`.

### 1. Préparation de l'environnement


In [ ]:
import os
import sys
import pandas as pd

pd.set_option('display.max_columns', None)
sys.path.append(os.path.abspath(os.path.join('..')))
import src.utils_viz as uv

### 2. Chargement du dataset enrichi


In [ ]:
df = pd.read_csv("../data/processed/football_ml_dataset.csv")
print(f"Dataset chargé pour visualisation : {df.shape}")

### 3. Traçage et sauvegarde de la Distribution


In [ ]:
fig_dist = uv.plot_target_distribution(df)

# Création du dossier pour stocker les figures du rapport Quarto s'il n'existe pas
os.makedirs("../reports/figures/", exist_ok=True)
fig_dist.savefig("../reports/figures/distribution_density.png", dpi=300)

### 4. Traçage et sauvegarde des Corrélations


In [ ]:
# On sélectionne les colonnes numériques les plus pertinentes pour le scouting
features_numeriques = [
    'market_value_in_eur', 'highest_market_value_in_eur', 'age', 
    'height_in_cm', 'goals', 'assists', 'minutes_played', 
    'goals_per_90', 'assists_per_90', 'international_caps'
]

fig_corr = uv.plot_correlation_matrix(df, features_numeriques)
fig_corr.savefig("../reports/figures/correlation_matrix.png", dpi=300)

print("\n[SUCCÈS] Les graphiques ont été générés et exportés dans 'reports/figures/' !")

### Profils et Distributions Caractéristiques (fig-distribution-density)

L'analyse de la distribution brute de la valeur marchande (market_value_in_eur) met en évidence une asymétrie positive extrême (skewness élevée). La très grande majorité des joueurs professionnels possède une valeur marchande "modeste" (inférieure à $5$ millions d'euros), tandis qu'une infime minorité d'athlètes d'élite (les "outliers" du marché comme Mbappé ou Haaland) concentre des valeurs record dépassant les $100$ millions d'euros.

Pour stabiliser la variance et rendre cette variable exploitable par nos futurs algorithmes de Machine Learning (notamment pour éviter que les modèles ne soient biaisés par les super-stars), une transformation logarithmique $\log(1 + x)$ a été testée (Figure de droite). Elle permet d'obtenir une distribution beaucoup plus proche d'une loi normale, configuration idéale pour les critères de convergence des modèles prédictifs.

### Corrélations Globales (fig-correlation)

L'examen de la matrice de corrélation de Pearson révèle plusieurs relations statistiques fondamentales :

- Performance quantitative vs Valeur : Les variables cumulées goals, assists, et surtout international_caps (sélections nationales) affichent une corrélation positive forte avec la valeur marchande. L'exposition internationale agit comme un multiplicateur de valeur économique.

- Le facteur de l'Âge : La variable age présente une corrélation linéaire faible lorsqu'elle est prise globalement, ce qui suggère une relation non-linéaire (courbe en cloche ou parabolique). En effet, la valeur croît lors de la post-formation, stagne à la maturité (24-28 ans), puis décroît fortement à l'approche de la trentaine.

- Biais mécanique : On note une corrélation presque parfaite entre market_value_in_eur et highest_market_value_in_eur, confirmant que le statut historique d'un joueur dicte durablement son plancher d'évaluation financière sur le marché.

---

# Analyse Exploratoire des Données (EDA) {#sec-eda}

Dans cette section, nous analysons les relations statistiques fondamentales qui régissent votre domaine d'étude au sein du jeu de données.

## Chapitre 4 : Travaux Pratiques d'Exploration (EDA)


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 🔎 Étape 3 : Analyse Exploratoire des Données (EDA) (Squelette Étudiant)

Cette étape correspond au troisième chapitre du cours. L'objectif est d'explorer et de résumer les propriétés statistiques fondamentales de vos données et de réaliser du **Feature Engineering** pour enrichir vos modèles.

### 1. Préparation de l'environnement


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration Pandas
pd.set_option('display.max_columns', None)

# Chargement du dataset processed
df = pd.read_csv("../data/processed/football_ml_dataset.csv")
print(f"Dataset initial : {df.shape}")

### 2. FEATURE ENGINEERING (Ingénierie de variables)


In [ ]:
print("Création de variables dérivées avancées...")

# 1. Variable cyclique ou catégorielle sur l'expérience (Matrice de maturité)
# L'âge a un effet parabolique. On capture cela en créant aussi une variable au carré.
df['age_squared'] = df['age'] ** 2

# 2. Ratio d'efficacité internationale
# Buts par sélection pour mesurer l'impact réel en équipe nationale
df['intl_efficiency'] = df['international_goals'] / (df['international_caps'] + 1)

# 3. Ratio de déclin / dynamique financière
# Différence relative entre la valeur actuelle et la valeur maximale historique
df['value_drop_ratio'] = (df['highest_market_value_in_eur'] - df['market_value_in_eur']) / (df['highest_market_value_in_eur'] + 1)

# 4. Indicateur de discipline (Cartons par minute jouée)
df['cards_per_90'] = ((df['yellow_cards'] + (df['red_cards'] * 2)) / (df['minutes_played'] + 1)) * 90

print(f"Dataset après Feature Engineering : {df.shape}")

### 3. ANALYSE VISUELLE AVANCÉE (Insight Majeur 3)


In [ ]:
# Analyse de la valeur marchande selon l'âge et le poste occupé
plt.figure(figsize=(12, 7))
sns.lineplot(data=df, x='age', y='market_value_in_eur', hue='position', errorbar=None, linewidth=2.5)
plt.title("Évolution de la Valeur Marchande Moyenne par Âge et Position", fontsize=14)
plt.xlabel("Âge du joueur")
plt.ylabel("Valeur Marchande Moyenne (EUR)")
plt.grid(True, linestyle='--', alpha=0.6)

# Sauvegarde de la figure pour le rapport Quarto
os.makedirs("../reports/figures/", exist_ok=True)
plt.savefig("../reports/figures/age_position_valuation.png", dpi=300)
plt.show()

### 4. SAUVEGARDE DU DATASET DE MODÉLISATION


In [ ]:
# On écrase ou on crée un fichier final prêt à être ingéré par le Machine Learning
df.to_csv("../data/processed/football_final_features.csv", index=False)
print("[SUCCÈS] Dataset final sauvegardé !")

# Modélisation et Apprentissage {#sec-modelling}

Le pipeline complet intègre à la fois la branche analytique tabulaire (Machine Learning) et la branche d'analyse visuelle ou de signaux complexes (Deep Learning CNN) :

```mermaid
graph TD
    A[Données Brutes Multi-Sources CSV/API] -->|Formatage & Alignement| B(data_clean.clean_dates)
    C[Données Externes Complémentaires] -->|Imputation & Interpolation| D(data_clean.impute_missing_values)
    B & D -->|Gestion Outliers| E[Jeu de données Propre & Fusionné]
    E -->|Extraction Temporelle/Caractéristiques| F[Feature Engineering]
    F -->|Splits Temporels ou Stratifiés| G[Modèle Machine Learning Tabulaire]
    H[Flux Multimédias Réels Images/Signaux] -->|Prétraitement d'images/signaux| I[Réseau Convolutif CNN TensorFlow]
    G -->|Prédictions de la Problématique Métier| J[Livrables & Aide à la Décision]
    I -->|Détection de Motifs Complexes| J
    
    style E fill:#e0f2fe,stroke:#0284c7,stroke-width:2px
    style J fill:#f0fdf4,stroke:#16a34a,stroke-width:2px
    style G fill:#fef3c7,stroke:#d97706,stroke-width:2px
    style I fill:#fef3c7,stroke:#d97706,stroke-width:2px
```

## Chapitre 5 : Travaux Pratiques de Modélisation (ML & DL)


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 🧠 Étape 5 : Modélisation (Machine Learning & Deep Learning) (Squelette Étudiant)

Cette étape correspond au cinquième chapitre du cours. L'objectif est d'implémenter d'une part un modèle de Machine Learning tabulaire (ex: RandomForest) et d'autre part un réseau de neurones convolutif (CNN) sous TensorFlow pour traiter des images ou signaux complexes.

### 1. Préparation de l'environnement


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
import tensorflow as tf
from tensorflow.keras import layers, models

sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Librairies de modélisation importées avec succès !")
print("Version TensorFlow :", tf.__version__)

### 2. Modélisation Tabulaire (Machine Learning)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Entraînez un modèle d'apprentissage supervisé (ex: forêt aléatoire) sur les caractéristiques extraites de votre jeu de données.


In [ ]:
# Chargement des données propres et ingénierie des caractéristiques
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df_feat = dc.feature_engineering(df, 'timestamp')

features = ['hour', 'dayofweek']
target = 'value'

# Split chronologique simple pour l'entraînement
X_train = df_feat[features].iloc[:-4]
y_train = df_feat[target].iloc[:-4]

# TODO: Instancier et entraîner RandomForestRegressor sur (X_train, y_train)
rf_model = RandomForestRegressor(n_estimators=10, random_state=42)
rf_model.fit(X_train, y_train)
print("Modèle de forêt aléatoire entraîné !")

### 3. Modélisation Vision / Deep Learning (CNN & TensorFlow)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Pour des motifs complexes (images/signaux), mettez en place un réseau convolutif (Conv2D, Pooling, Dense) pour classifier ou enrichir vos prédictions.


In [ ]:
# Génération fictive d'un jeu d'images simples (64x64 pixels) de cercles (Classe 0) vs rectangles (Classe 1)
def generate_dummy_images(num_samples=100):
    images = np.zeros((num_samples, 64, 64, 3), dtype=np.float32)
    labels = np.zeros(num_samples, dtype=np.int32)
    for i in range(num_samples):
        label = np.random.choice([0, 1])
        labels[i] = label
        images[i, :, :, :] = 0.2 + np.random.normal(0, 0.01, (64, 64, 3))
        if label == 1:
            images[i, 10:30, 10:30, 0] = 0.8
        else:
            images[i, 20:40, 20:40, 1] = 0.8
    return images, labels

X_images, y_labels = generate_dummy_images(100)
split = int(0.8 * len(X_images))
X_img_train = X_images[:split]
y_img_train = y_labels[:split]

print(f"Dataset d'images brutes généré. Dimensions Train : {X_img_train.shape}")

In [ ]:
# TODO: Définir l'architecture séquentielle du CNN avec layers.Conv2D et layers.MaxPooling2D
cnn_model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

cnn_model.summary()

In [ ]:
# TODO: Compiler et entraîner le CNN avec l'optimiseur adam et une binary_crossentropy
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_img_train, y_img_train, epochs=2, batch_size=32, verbose=1)
print("CNN entraîné avec succès !")

---

# Évaluation Métrique et Validation {#sec-evaluation}

## Chapitre 6 : Travaux Pratiques d'Évaluation & Robustesse


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 🧪 Étape 6 : Évaluation Métrique & Robustesse (Squelette Étudiant)

Cette étape correspond au sixième chapitre du cours. L'objectif est de mettre en place un protocole d'évaluation rigoureux (splits d'évaluation adaptés) et de calculer les métriques clés de performance pour valider scientifiquement la qualité de vos modèles.

### 1. Préparation de l'environnement


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Librairies prêtes pour l'évaluation des modèles !")

### 2. Évaluation du modèle Tabulaire

**À COMPLÉTER PAR L'ÉTUDIANT :**
Calculez et interprétez les métriques d'erreur sur vos prédictions (MAE, RMSE, R²).


In [ ]:
# Chargement des données et ré-entraînement rapide pour évaluation
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df_feat = dc.feature_engineering(df, 'timestamp')

features = ['hour', 'dayofweek']
target = 'value'

X_train = df_feat[features].iloc[:-4]
y_train = df_feat[target].iloc[:-4]
X_test = df_feat[features].iloc[-4:]
y_test = df_feat[target].iloc[-4:]

# Entraînement
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=10, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# TODO: Calculez MAE, RMSE et R² entre y_test et y_pred
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test MAE  : {mae:.3f}")
print(f"Test RMSE : {rmse:.3f}")
print(f"Test R²   : {r2:.3f}")

### 3. Protocole de Validation Croisée (Out-of-Fold / Chronologique)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Décrivez et codez (ou documentez) une stratégie de validation croisée adaptée au comportement temporel de vos données pour valider la robustesse de votre modèle sans fuite d'information.


In [ ]:
# TODO: Proposer un script de K-Fold temporel (TimeSeriesSplit) ou de validation croisée classique
print("Protocole de validation documenté avec succès !")

---

# Data Storytelling et Communication {#sec-storytelling}

## Chapitre 7 : Travaux Pratiques de Storytelling


In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 📢 Étape 7 : Data Storytelling & Communication (Squelette Étudiant)

Cette étape correspond au septième et dernier chapitre de data science. L'objectif est de synthétiser vos résultats pour des profils métiers ou décideurs et de proposer des visualisations interactives ou dynamiques pour valoriser vos conclusions.

### 1. Préparation de l'environnement


In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))

print("Librairies prêtes pour la phase de Data Storytelling !")

### 2. Synthèse métier et Storytelling

**À COMPLÉTER PAR L'ÉTUDIANT :**
Traduisez vos métriques techniques en impacts stratégiques (par exemple, gains financiers, réduction de coûts, amélioration de la sécurité, etc.).


In [ ]:
# TODO: Écrire des exemples de recommandations stratégiques
print("Storytelling métier initialisé avec succès.")

::: {.content-visible unless-format="pdf"}
<div id="plotly-0d78b9ba-e321-406f-aaa7-5eb174c5b5d1" style="width:100%; height:400px; background: white; border-radius: 8px;"></div>
<script type="text/javascript">
  document.addEventListener("DOMContentLoaded", function() {
    if (typeof Plotly !== 'undefined') {
      Plotly.newPlot('plotly-0d78b9ba-e321-406f-aaa7-5eb174c5b5d1', [{"type": "scatter", "x": [1, 2, 3], "y": [10, 15, 13], "mode": "lines+markers", "name": "Donn\u00e9es de Test"}], {"title": "Mon Graphique Plotly de Test"}, {"responsive": true});
    } else {
      console.error("Plotly library is not loaded.");
    }
  });
</script>
:::

### 3. Visualisation Interactive (Plotly)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Générez un graphique interactif (par exemple en utilisant Plotly ou des éléments OJS dans le document final) pour permettre aux décideurs d'interagir dynamiquement avec vos données.


In [ ]:
# TODO: Créer un graphique interactif simple
print("Module de communication dynamique prêt !")

::: {.content-visible unless-format="pdf"}
<div id="plotly-5ee76a01-54c1-4919-818b-0ffc98f8133b" style="width:100%; height:400px; background: white; border-radius: 8px;"></div>
<script type="text/javascript">
  document.addEventListener("DOMContentLoaded", function() {
    if (typeof Plotly !== 'undefined') {
      Plotly.newPlot('plotly-5ee76a01-54c1-4919-818b-0ffc98f8133b', [{"type": "scatter", "x": [1, 2, 3], "y": [10, 15, 13], "mode": "lines+markers", "name": "Donn\u00e9es de Test"}], {"title": "Mon Graphique Plotly de Test"}, {"responsive": true});
    } else {
      console.error("Plotly library is not loaded.");
    }
  });
</script>
:::




## Présentation des Résultats (Livrables Interactifs)

::: {.panel-tabset}

### 📺 Diaporama de Soutenance (RevealJS)
Ci-dessous est intégré le squelette de votre diaporama de soutenance RevealJS. Utilisez-le pour présenter votre démarche aux décideurs de façon professionnelle.

<iframe src="slides.html" width="100%" height="500px" style="border: 1px solid #e2e8f0; border-radius: 8px; background: white;"></iframe>

### 📊 Exemple de Dashboard Dynamique (OJS / Plotly)
::: {.content-visible unless-format="pdf"}
Voici un exemple minimal de code montrant comment intégrer un graphique dynamique contrôlé par un composant d'interface utilisateur en Observable JS (OJS).




```{ojs}
//| echo: true
// Boutons de sélection interactifs OJS
viewof selectedCategory = Inputs.select(["Toutes", "A", "B", "C"], {value: "Toutes", label: "Filtrer par Catégorie :"})
```

```{ojs}
//| echo: false
// Données simulées réactives
data = [
  {timestamp: "2026-05-18T00:00:00Z", value: 10.5, category: "A"},
  {timestamp: "2026-05-18T02:00:00Z", value: 12.1, category: "A"},
  {timestamp: "2026-05-18T04:00:00Z", value: 14.7, category: "A"},
  {timestamp: "2026-05-18T05:00:00Z", value: 15.2, category: "A"},
  {timestamp: "2026-05-18T06:00:00Z", value: 16.0, category: "B"},
  {timestamp: "2026-05-18T07:00:00Z", value: 18.3, category: "B"},
  {timestamp: "2026-05-18T09:00:00Z", value: 21.5, category: "B"},
  {timestamp: "2026-05-18T10:00:00Z", value: 22.0, category: "B"},
  {timestamp: "2026-05-18T12:00:00Z", value: 25.4, category: "C"},
  {timestamp: "2026-05-18T13:00:00Z", value: 26.1, category: "C"},
  {timestamp: "2026-05-18T15:00:00Z", value: 28.9, category: "C"},
  {timestamp: "2026-05-18T16:00:00Z", value: 30.2, category: "C"}
]

// Filtrage réactif de la donnée
filteredData = selectedCategory === "Toutes" 
  ? data 
  : data.filter(d => d.category === selectedCategory)

// Tracé interactif avec la librairie Plotly
Plotly.newPlot('dynamic-chart', [{
  x: filteredData.map(d => d.timestamp),
  y: filteredData.map(d => d.value),
  type: 'scatter',
  mode: 'lines+markers',
  marker: {color: '#1A73E8', size: 8},
  line: {shape: 'spline', color: '#1A73E8', width: 3}
}], {
  title: 'Évolution Dynamique des Valeurs (Filtrée)',
  margin: {t: 50, b: 50, l: 50, r: 50},
  paper_bgcolor: 'rgba(0,0,0,0)',
  plot_bgcolor: 'rgba(0,0,0,0)',
  xaxis: {gridcolor: '#E5E7EB'},
  yaxis: {gridcolor: '#E5E7EB'}
})
```




::: {#dynamic-chart style="width:100%; height:400px; background: white; border-radius: 8px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);"}
:::
:::

:::

---

# Utilisation de l'Intelligence Artificielle {#sec-ai}

Dans une démarche de transparence scientifique et académique, cette section détaille la manière dont les outils d'Intelligence Artificielle (IA) générative ont été intégrés tout au long de la réalisation de ce projet.

## Cartographie de l'utilisation de l'IA

| Outil d'IA | Cas d'usage (Pourquoi ?) | Méthode d'utilisation (Comment ?) | Rôle et Validation Humaine |
| :--- | :--- | :--- | :--- |
| **[Outil d'IA]** | *[À compléter par les étudiants]* | *[À compléter par les étudiants]* | *[À compléter par les étudiants]* |

## Principes de Rigueur et Responsabilité

1. **Responsabilité intellectuelle** : L'équipe assume l'entière responsabilité des analyses, des choix de modèles et des conclusions présentées dans ce rapport.
2. **Lutte contre les hallucinations** : Chaque suggestion technique a fait l'objet d'une validation empirique.
3. **Protection des données** : Aucun jeu de données confidentiel ou sensible n'a été soumis à des modèles tiers en ligne.

---

# Bibliographie {.unnumbered}

::: {#refs}
:::